### Data Pull
Find the nearest coordinates containing all the necessary datasets and it's index value

Upload CSV file containing: "location_name, latitude, longitude, file_path"

In [39]:
import pandas as pd

locations = pd.read_csv("ATON_Locations.csv")
display(locations)

location = locations.iloc[0]
name = location["location_name"]
lat = locations["latitude"]
lon = locations["longitude"]
file_path = location["file_name"]

,location_name,latitude,longitude,file_name
0,Belle,29.071107,-90.227879,/nrel/US_wave/v1.0.1/Gulf_of_Mexico_and_Puerto...
1,Freeport,28.912614,-95.269769,/nrel/US_wave/v1.0.1/Gulf_of_Mexico_and_Puerto...
2,Galveston,29.266923,-94.571306,/nrel/US_wave/v1.0.1/Gulf_of_Mexico_and_Puerto...
3,Desdemona_Sands,46.225134,-123.973980,/nrel/US_wave/West_Coast/West_Coast_wave_2010.h5
4,Clatsop_Spit,46.248895,-124.011479,/nrel/US_wave/West_Coast/West_Coast_wave_2010.h5
5,Columbia_River,46.185001,-124.183709,/nrel/US_wave/West_Coast/West_Coast_wave_2010.h5
6,Chesapeake_Bay_Mouth,36.825459,-75.781913,/nrel/US_Wave/Atlantic/Atlantic_wave_2010.h5
7,Chesapeake_Bay_Southern,36.872679,-75.868236,/nrel/US_Wave/Atlantic/Atlantic_wave_2010.h5
8,Cape_Henry,36.885472,-75.888675,/nrel/US_Wave/Atlantic/Atlantic_wave_2010.h5


In [ ]:
#File Locations

# West Coast (file)
#"/nrel/US_wave/West_Coast/West_Coast_wave_2010.h5",
#    "r"

#file = h5pyd.File("/nrel/US_wave/West_Coast/West_Coast_wave_2010.h5")


# Gulf of Mexico (file)
#"/nrel/US_wave/v1.0.1/Gulf_of_Mexico_and_Puerto_Rico/GOM_PR_2010.h5",
#       "r"

#file = h5pyd.File("/nrel/US_wave/v1.0.1/Gulf_of_Mexico_and_Puerto_Rico/GOM_PR_2010.h5")


# Virginia Beach
#"/nrel/US_Wave/Atlantic/Atlantic_wave_2010.h5",
#       "r"

#file = h5pyd.File("/nrel/US_wave/Atlantic/Atlantic_wave_2010.h5")




Check what regions exist in NREL US Wave Folder

In [ ]:
## Checks what regions exist in NREL US Wave folder
# import h5pyd

# folder = h5pyd.Folder("/nrel/US_wave/")

# print("Available regions:")
# for name in folder:
#     print(name)


In [24]:
# # Check contents of selected folders
# import h5pyd

# for version in ["v1.0.1"]:
#     print(f"\nContents of {version}:")

#     try:
#         folder = h5pyd.Folder(f"/nrel/US_wave/{version}/")
#         for name in folder:
#             print(name)
#     except Exception as e:
#         print(type(e).__name__, str(e))

In [25]:
# # Outputs folder names showing what years of data are available
# import h5pyd

# folder = h5pyd.Folder("/nrel/US_wave/v1.0.1/Gulf_of_Mexico_and_Puerto_Rico/")

# for name in folder:
#     print(name)

# gulf_folder = h5pyd.Folder(
#     "/nrel/US_wave/v1.0.1/Gulf_of_Mexico_and_Puerto_Rico/GOM_PR_2010.h5/"
# )

# for name in gulf_folder:
#     print(name)

In [ ]:
# import h5pyd

# file = h5pyd.File("/nrel/US_wave/Atlantic/Atlantic_wave_2010.h5")

# for name in file:
#     print(name)

In [36]:
# Checks connection to the file and displays available datasets
import h5pyd

try:
    f = h5pyd.File(
        "/nrel/US_wave/v1.0.1/Gulf_of_Mexico_and_Puerto_Rico/GOM_PR_2010.h5",
        "r"
    )
    print("Connection successful!")
    print(list(f.keys()))
    f.close()

except Exception as e:
    print("Error type:", type(e).__name__)
    print("Error message:", str(e))

Connection successful!
['coordinates', 'depth', 'direction_of_maximum_directionally_resolved_wave_power', 'directionality_coefficient', 'energy_period', 'mean_absolute_period', 'mean_wave_direction', 'mean_zero-crossing_period', 'meta', 'missing_time_index', 'omnidirectional_wave_power', 'peak_period', 'peak_wave_direction', 'significant_wave_height', 'spectral_width', 'time_index']


In [28]:

import h5pyd

with h5pyd.File(
    "/nrel/US_wave/Atlantic/Atlantic_wave_2010.h5",
    "r"
) as f:

    ds = f["directionality_coefficient"]

    print("Shape:", ds.shape)
    print("Data type:", ds.dtype)
    print("Scale factor:", ds.attrs.get("scale_factor"))
    print("First dimension length:", ds.shape[0])


Shape: (2920, 2635135)
Data type: float32
Scale factor: None
First dimension length: 2920


In [ ]:

import h5pyd

with h5pyd.File(
    "/nrel/US_wave/Atlantic/Atlantic_wave_2010.h5",
    "r"
) as f:

    print("Available datasets:")
    for key in f.keys():
        print(key)

    print("\nCoordinates dataset:")
    if "coordinates" in f:
        coords = f["coordinates"]
        print("Shape:", coords.shape)
        print("First 5 coordinates:")
        print(coords[:5])


In [ ]:

import h5pyd
import numpy as np

lat = 36.885472
lon = -75.888675


with h5pyd.File(
    "/nrel/US_wave/Atlantic/Atlantic_wave_2010.h5",
    "r"
) as f:

    coords = f["coordinates"][:]

    # Approximate geographic distance
    distance = (
        (coords[:, 0] - lat)**2
        + (np.cos(np.radians(lat)) *
           (coords[:, 1] - lon))**2
    )

    idx = np.argmin(distance)

    print("Nearest grid point index:", idx)
    print("Nearest coordinates:", coords[idx])


Nearest grid point index: 1687429
Nearest coordinates: [ 36.8858 -75.888 ]


In [31]:
import pandas as pd

with h5pyd.File(
    "/nrel/US_wave/Atlantic/Atlantic_wave_2010.h5",
    "r"
) as f:

    directionality = f[
        "directionality_coefficient"
    ][:, idx]

    time = pd.to_datetime(
        f["time_index"][:].astype(str),
        utc=True
    )

df_directionality = pd.DataFrame(
    {"directionality_coefficient": directionality},
    index=time
)

print(df_directionality.head())
print("Number of measurements:", len(df_directionality))


parameters = [
    "significant_wave_height",
    "peak_period",
    "energy_period",
    "omni-directional_wave_power",
    "mean_wave_direction",
    "spectral_width",
    "directionality_coefficient"
]

with h5pyd.File(
    "/nrel/US_wave/Atlantic/Atlantic_wave_2010.h5",
    "r"
) as f:

    wave_data = {}

    for parameter in parameters:
        wave_data[parameter] = f[parameter][:, idx]

df = pd.DataFrame(wave_data, index=time)

display(df.head())


                           directionality_coefficient
2010-01-01 00:00:00+00:00                       0.853
2010-01-01 03:00:00+00:00                       0.886
2010-01-01 06:00:00+00:00                       0.839
2010-01-01 09:00:00+00:00                       0.721
2010-01-01 12:00:00+00:00                       0.732
Number of measurements: 2920


,significant_wave_height,peak_period,energy_period,omni-directional_wave_power,mean_wave_direction,spectral_width,directionality_coefficient
2010-01-01 00:00:00+00:00,0.55456,8.3279,4.7676,1066.897949,51.299999,0.3726,0.853
2010-01-01 03:00:00+00:00,0.49696,6.3268,4.6177,792.998291,47.261002,0.3799,0.886
2010-01-01 06:00:00+00:00,0.49022,5.2677,4.4799,752.196289,52.509998,0.4235,0.839
2010-01-01 09:00:00+00:00,0.54653,5.2677,4.3757,905.136597,76.600998,0.4316,0.721
2010-01-01 12:00:00+00:00,0.69917,6.9337,4.1204,1580.308228,104.471001,0.3928,0.732


In [32]:

import h5pyd
import pandas as pd
import numpy as np

#idx = 413889

parameters = [
    "significant_wave_height",
    "peak_period",
    "energy_period",
    "omni-directional_wave_power",
    "mean_wave_direction",
    "spectral_width",
    "directionality_coefficient"
]

with h5pyd.File(
    "/nrel/US_wave/Atlantic/Atlantic_wave_2010.h5",
    "r"
) as f:

    wave_data = {}

    for parameter in parameters:
        wave_data[parameter] = f[parameter][:, idx]

    time = pd.to_datetime(
        f["time_index"][:].astype(str),
        utc=True
    )

df = pd.DataFrame(wave_data, index=time)

display(df.head())
print("Number of measurements:", len(df))


,significant_wave_height,peak_period,energy_period,omni-directional_wave_power,mean_wave_direction,spectral_width,directionality_coefficient
2010-01-01 00:00:00+00:00,0.55456,8.3279,4.7676,1066.897949,51.299999,0.3726,0.853
2010-01-01 03:00:00+00:00,0.49696,6.3268,4.6177,792.998291,47.261002,0.3799,0.886
2010-01-01 06:00:00+00:00,0.49022,5.2677,4.4799,752.196289,52.509998,0.4235,0.839
2010-01-01 09:00:00+00:00,0.54653,5.2677,4.3757,905.136597,76.600998,0.4316,0.721
2010-01-01 12:00:00+00:00,0.69917,6.9337,4.1204,1580.308228,104.471001,0.3928,0.732


Number of measurements: 2920


In [33]:
df.to_csv("Chesapeake_Bay_Mouth_wave_data_2010.csv")

In [34]:
print(df.isna().sum())
print(df.describe())

significant_wave_height        0
peak_period                    0
energy_period                  0
omni-directional_wave_power    0
mean_wave_direction            0
spectral_width                 0
directionality_coefficient     0
dtype: int64
       significant_wave_height  peak_period  energy_period  \
count              2920.000000  2920.000000    2920.000000   
mean                  1.136764     8.172534       4.724972   
std                   0.617123     3.135444       1.615378   
min                   0.175870     1.923100       1.727100   
25%                   0.714343     5.267700       3.564675   
50%                   0.960850     8.327900       4.234400   
75%                   1.385608    10.002400       5.413100   
max                   4.639320    20.814800      10.374100   

       omni-directional_wave_power  mean_wave_direction  spectral_width  \
count                  2920.000000          2920.000000     2920.000000   
mean                   6990.326660           11